# Driver Monitoring Using Multi-Modal Fusion: training on Google Colab

Runs the leave-one-subject-out experiments on a Colab GPU. Results are written to
Google Drive after every fold, so if Colab disconnects, run the cells again and
training continues from the last finished fold.

**Before you start**

1. *Runtime → Change runtime type → T4 GPU*.
2. In Google Drive, create the folder `MyDrive/driver-monitoring/` and upload
   `drozy_processed.zip` into it (made on the training PC; it holds the processed
   DROZY windows and the Teacher checkpoints).
   DROZY is for research use only: keep this folder private, do not share it.
3. If the GitHub repository is private: click the key icon (*Secrets*) on the left,
   add a secret named `GITHUB_TOKEN` holding a GitHub personal access token with
   read access to the repository, and allow this notebook to use it.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/driver-monitoring"
RUNS = f"{DRIVE}/runs"
!mkdir -p "{RUNS}"
!ls -la "{DRIVE}"

## 3. Get the code

In [ ]:
import os, subprocess
REPO = "lomoropaul/driver-monitoring-using-multi-modal-fusion"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.exists("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
%cd /content/repo
!git log --oneline -1
!pip install -q -e . --no-deps

## 4. Unpack the data

Copies the processed data from Drive to the Colab disk (much faster to read than
Drive), and puts the Teacher checkpoints in the Drive runs folder.

In [ ]:
!mkdir -p /content/data
!unzip -q -n "{DRIVE}/drozy_processed.zip" -d /content/data
!test -d "{RUNS}/teacher" || cp -r /content/data/runs/teacher "{RUNS}/teacher"
!ls /content/data/processed "{RUNS}"
DATA = "/content/data/processed"

## 5. Train the mid-size Student

Same early-fusion architecture as the original fusion model (57,078 parameters,
about 74% smaller than the Teacher).

* `scratch`: the Student without distillation (ablation)
* `student`: the Student distilled from the Teacher, with the (T, alpha) grid chosen per fold on validation

Each finished fold is saved to Drive. If Colab disconnects, run sections 2-4 again
and then this cell: finished folds are loaded, not retrained.

In [ ]:
!python scripts/train_distill.py --stage scratch --student student_mid --data {DATA} --runs "{RUNS}"

In [ ]:
!python scripts/train_distill.py --stage student --student student_mid --data {DATA} --runs "{RUNS}"

## 6. Compare with the Teacher

Latency is skipped here: Colab's CPU says nothing about the Jetson.
The full comparison with all models is run on the training PC after copying
`runs/` back from Drive.

In [ ]:
!python scripts/compare_runs.py --runs "{RUNS}" --no-latency